# Build Your Own AI StudyBot with Google ADK
### A beginner-friendly codelab

In this notebook you will build **StudyBot**, an AI assistant for students. StudyBot can:
- Explain any topic at a beginner, intermediate, or advanced level
- Quiz you with multiple-choice questions
- Build a day-by-day study plan for an exam

This codelab follows the StudyBot code that is currently in this repository. The live agent uses **Gemini 2.5 Flash Lite** in the code below, but the same pattern works with other Gemini models if you want to swap them later.

By the end, you will know how to build and run a working ADK agent in your browser.

## What You Will Learn

You will learn how to:
- Set up Google ADK in a Python environment
- Create the files StudyBot needs
- Write an ADK agent in `studybot/agent.py`
- Add tools in `studybot/tools.py`
- Run the agent with `adk web`
- Customize the agent's personality and behavior

### What is an AI agent?

A normal chatbot only answers messages. An AI agent can answer messages and also use tools. That means it can follow instructions, call functions, and help with tasks like quizzes and study plans.

### What is Google ADK?

Google ADK (Agent Development Kit) is an open-source Python framework for building AI agents. It handles the agent loop, tool calling, and session management so you can focus on what the agent should do instead of the plumbing.

## Before You Start

You will need:
- A Google account
- A GitHub account
- This repository open in GitHub Codespaces or VS Code
- A Gemini API key

### A quick note for beginners

You do not need to understand everything before you begin. Follow the steps in order. Each one builds on the last. If you get stuck, the common error notes in the notebook should help you recover quickly.

## Step 1 — Set Up Google ADK

Open a terminal and install ADK. If you are working inside a notebook, you can also run the commands from a code cell.

After installation, verify that the `adk` command is available.

In [ ]:
# Install dependencies from this repository
!pip install -r requirements.txt

# Verify the ADK CLI installation
!adk --version

### Common problems at this step

- If `adk` is not found, rerun the install command and wait for it to finish.
- If `pip` is not found, try `pip3 install google-adk`.
- If you get a permissions error, add `--user` to the install command.

## Step 2 — Get Your API Key and Create the Project Files

Go to https://aistudio.google.com/apikey, sign in, and create a Gemini API key. Copy it somewhere safe.

Next, create the files StudyBot will use. If they already exist in your workspace, just open them instead of creating duplicates.

You will create:
- `studybot/agent.py`
- `studybot/tools.py`
- `studybot/__init__.py`
- `studybot/.env`
- `.gitignore`

In [ ]:
# Create the project structure (cross-platform)
from pathlib import Path

Path("studybot").mkdir(parents=True, exist_ok=True)
for file_path in [
    "studybot/agent.py",
    "studybot/__init__.py",
    "studybot/tools.py",
    "studybot/.env",
    ".gitignore",
]:
    Path(file_path).touch(exist_ok=True)

print("Files are ready.")

Copy the sample env file first:

```bash
cp studybot/.env.example studybot/.env
```

Then open `studybot/.env` and set:

```
GOOGLE_API_KEY=paste_your_key_here
GOOGLE_GENAI_USE_VERTEXAI=FALSE
```

Optional:

```
STUDYBOT_MODEL=gemini-2.5-flash-lite
```

Open `.gitignore` and make sure these lines exist:

```
studybot/.env
.venv/
venv/
__pycache__/
*.pyc
studybot/.adk/
```

## Step 3 — Write the Agent

The agent lives in `studybot/agent.py`. ADK looks for a variable named exactly `root_agent`.

This version loads your `.env` file, imports all three tools, and sets up the agent with a simple helpful instruction.

In [ ]:
# Paste this into studybot/agent.py
import os
from pathlib import Path
from google.adk.agents import Agent
from studybot.tools import explain_topic, quiz_student, study_planner

def _load_local_env() -> None:
    env_path = Path(__file__).with_name(".env")
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key:
            os.environ.setdefault(key, value)

_load_local_env()

root_agent = Agent(
    model="gemini-2.5-flash-lite",
    name="StudyBot",
    description="A helpful assistant for students to answer questions and provide explanations.",
    instruction="""You are StudyBot, an AI assistant designed to help students. 
    When a student asks you to explain something, use the explain_topic tool.
    Always be encouraging and clear.""",
    tools=[explain_topic, quiz_student, study_planner],
)



### What each part does

- `Path(__file__).with_name(".env")` finds the `.env` file next to `agent.py`.
- `_load_local_env()` reads your local key and places it into the environment.
- `root_agent` is the object ADK uses to start your assistant.
- `tools=[...]` tells the agent which functions it is allowed to call.

## Step 4 — Add the First Tool: `explain_topic`

Tools are normal Python functions with clear docstrings. The docstring matters because the model reads it to understand what the tool does and what arguments it expects.

In [ ]:
# Paste this into studybot/tools.py
def explain_topic(topic: str, level: str = "beginner") -> str:
    """Explains a study topic at a given difficulty level.

    Args:
        topic: The subject or concept the student wants to understand.
        level: The difficulty level - either 'beginner', 'intermediate', or 'advanced'.

    Returns:
        A structured explanation of the topic at the requested level.
    """
    return f"Please explain {topic} at a {level} level with examples and key points."


# Backward-compatible alias for older references with a typo.
def expalin_topic(topic: str, level: str = "beginner") -> str:
    return explain_topic(topic=topic, level=level)


### Update the agent for Step 4

If you are following along in the files directly, update the import in `studybot/agent.py` to:

```python
from studybot.tools import explain_topic
```

Then update the tool list to:

```python
tools=[explain_topic],
```

Restart the agent and try: `Explain machine learning to me at a beginner level`

## Step 5 — Add the Quiz Tool

The quiz tool asks the model to generate exactly 3 multiple-choice questions and provide feedback after the student answers.

In [ ]:
# Add this below explain_topic in studybot/tools.py
def quiz_student(topic: str) -> str:
    """Generates a short quiz to test a student's knowledge on a topic.

    Args:
        topic: The subject the student wants to be quizzed on.

    Returns:
        A set of 3 multiple choice questions with options A, B, C, D.
    """
    return f"""Generate exactly 3 multiple choice questions about {topic}.
    Format each question like this:
    
    Q1: [question]
    A) [option]
    B) [option]  
    C) [option]
    D) [option]
    Answer: [correct letter]
    
    After the student answers, mark them and give encouraging feedback.""" 

### Update the agent for Step 5

Add the quiz tool to the import and tool list in `studybot/agent.py`:

```python
from studybot.tools import explain_topic, quiz_student
tools=[explain_topic, quiz_student],
```

Try: `Quiz me on Python functions`

## Step 6 — Add the Study Planner Tool

The study planner takes a subject and the number of days until the exam, then returns a day-by-day study schedule.

In [ ]:
# Add this below quiz_student in studybot/tools.py

def study_planner(subject: str, days_until_exam: int) -> str:
    """Creates a day-by-day study plan for a student preparing for an exam.

    Args:
        subject: The subject the student is preparing for.
        days_until_exam: The number of days the student has until their exam.

    Returns:
        A structured day-by-day study schedule with topics and goals.
    """
    return f"""Create a detailed {days_until_exam}-day study plan for {subject}.
    
    Format it exactly like this:
    
    Day 1: [Topic to cover]
    - Goal: [What the student should achieve]
    - Resources: [What to read or watch]
    - Practice: [Exercise to do]
    
    Repeat for each day. End with exam-day tips."""

### Update the agent for Step 6

Now the agent should import all three tools and use all three in the `tools` list:

```python
from studybot.tools import explain_topic, quiz_student, study_planner
tools=[explain_topic, quiz_student, study_planner],
```

Try: `Create a 5 day study plan for Python`

## Step 7 — Run the Agent

From the project root, start the ADK web app with:

```bash
adk web . --host 127.0.0.1 --port 8000 --allow_origins "regex:.*"
```

Then open the browser UI and start a new session. If a message does not get a response, turn token streaming off in the top-right corner and try again.

In [ ]:
# Optional but recommended: verify end-to-end response first
!python scripts/smoke_test.py

## Step 8 — Customize Your Agent

The `instruction` string controls personality and behavior. Try changing it to build a different assistant.

### Example: math tutor
```python
instruction="""You are MathBot, a patient math tutor.
You help students understand mathematics from basic arithmetic to calculus.
Always show step-by-step working and use simple examples."""
```

### Example: coding mentor
```python
instruction="""You are CodeMentor, a programming tutor.
You help students learn Python, JavaScript, and web development.
Always explain with working code examples."""
```

### Example: strict exam coach
```python
instruction="""You are ExamBot, a strict exam coach.
You only quiz students. Do not give hints or reveal answers early.
Only reveal the correct answer after the student has attempted."""
```

## Step 9 — Push to GitHub

Before pushing, make sure `.env` is ignored and your changes look correct.

Typical commands:
```bash
git status
git add .
git commit -m "feat: add StudyBot ADK codelab"
git push origin main
```

If Git says there is nothing to commit, save your files first and run the commands again.

## Where to Go Next

Once your StudyBot works, you can extend it with:
- Google Search
- More specialized subject tools
- A grading or feedback agent
- Cloud deployment

You now have the pattern for building a real agent: define the behavior, add tools, and let ADK coordinate the calls.